# Number of parents to leaves in the subject meta data
in this file i want to determine the number of parent nodes in the metadata that are adjacent to leaf nodes in the tree. 

In [2]:
import ast
import csv
from collections import Counter, defaultdict
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "metadata" / "subject_metadata.csv").exists():
            return cand
    raise FileNotFoundError("Could not find project root containing data/metadata/subject_metadata.csv")


def parse_chain(raw: str) -> list[int]:
    try:
        value = ast.literal_eval(raw)
    except (ValueError, SyntaxError):
        return []
    return [int(x) for x in value] if isinstance(value, list) else []


root = find_project_root()
subject_csv = root / "data" / "metadata" / "subject_metadata.csv"
question_csv = root / "data" / "metadata" / "question_metadata_task_1_2.csv"

# 1) Build subject tree
parent_by_child: dict[int, int] = {}
children_by_parent: dict[int, set[int]] = defaultdict(set)
all_subject_ids: set[int] = set()

with subject_csv.open("r", encoding="utf-8-sig", newline="") as fh:
    for row in csv.DictReader(fh):
        sid = int(row["SubjectId"])
        all_subject_ids.add(sid)
        p = row.get("ParentId", "")
        if p and p != "NULL":
            pid = int(p)
            parent_by_child[sid] = pid
            children_by_parent[pid].add(sid)

leaf_subject_ids = all_subject_ids - set(children_by_parent.keys())

# 2) For each question, count distinct parent nodes that have at least one leaf child in that question chain
counts = Counter()
max_count = 0
max_questions: list[int] = []

with question_csv.open("r", encoding="utf-8-sig", newline="") as fh:
    for row in csv.DictReader(fh):
        qid = int(row["QuestionId"])
        chain = set(parse_chain(row["SubjectId"]))

        parent_nodes_adjacent_to_leaves = {
            parent_by_child[sid]
            for sid in chain
            if sid in leaf_subject_ids and sid in parent_by_child and parent_by_child[sid] in chain
        }

        c = len(parent_nodes_adjacent_to_leaves)
        counts[c] += 1

        if c > max_count:
            max_count = c
            max_questions = [qid]
        elif c == max_count:
            max_questions.append(qid)

# 3) Report summary
n_questions = sum(counts.values())
print(f"Total questions analyzed: {n_questions:,}")
print(f"Maximum parent-nodes-adjacent-to-leaf count in any question: {max_count}")
print(f"Questions achieving maximum: {len(max_questions):,}")
print("\nDistribution:")
for k in sorted(counts.keys(), reverse=True):
    pct = 100 * counts[k] / n_questions
    print(f"  {k:2d} parent-adjacent nodes: {counts[k]:7,} questions ({pct:6.2f}%)")

Total questions analyzed: 27,613
Maximum parent-nodes-adjacent-to-leaf count in any question: 5
Questions achieving maximum: 2

Distribution:
   5 parent-adjacent nodes:       2 questions (  0.01%)
   4 parent-adjacent nodes:       8 questions (  0.03%)
   3 parent-adjacent nodes:     126 questions (  0.46%)
   2 parent-adjacent nodes:   1,292 questions (  4.68%)
   1 parent-adjacent nodes:  26,185 questions ( 94.83%)
